# BarqTrain Colab: Benchmark Comparison

This notebook compares four inference setups on the same prompt and output length:
- Baseline Hugging Face / PyTorch
- BarqTrain patch only
- BarqTrain patch with CUDA backend enabled
- Unsloth AI with full-weight precision (not 4-bit)

Methodology in this version:
- `ninja` is installed explicitly for native CUDA builds.
- Unsloth is installed using its official auto-install flow.
- Each setup is loaded, benchmarked, then cleaned up before the next run.
- The benchmark helper measures average latency, median latency, tokens/sec, and peak VRAM.


In [ ]:
%%bash
set -e
apt-get -y install python3-dev >/dev/null
pip -q install --upgrade pip
pip -q install ninja packaging pandas torch transformers datasets accelerate peft bitsandbytes
UNSLOTH_CMD="$(wget -qO- https://raw.githubusercontent.com/unslothai/unsloth/main/unsloth/_auto_install.py | python -)"
eval "$UNSLOTH_CMD" || true
if [ ! -d /content/BarqTrain ]; then
  git clone https://github.com/YASSERRMD/BarqTrain.git /content/BarqTrain
fi
cd /content/BarqTrain
pip -q install -e .
python - <<'PY'
import shutil
print('ninja:', shutil.which('ninja'))
PY


In [ ]:
%cd /content/BarqTrain

import gc
import importlib
import os
import statistics
import time

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
UNSLOTH_MODEL_NAME = "unsloth/Qwen2-0.5B-Instruct"
PROMPT = "Explain why fused kernels can improve LLM training throughput."
NEW_TOKENS = 64
WARMUP_RUNS = 2
MEASURE_RUNS = 5

def pick_dtype():
    if DEVICE != "cuda":
        return torch.float32
    if torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16

DTYPE = pick_dtype()

print("Device:", DEVICE)
print("Base model:", BASE_MODEL_NAME)
print("Unsloth model:", UNSLOTH_MODEL_NAME)
print("DType:", DTYPE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: CUDA not available. CUDA and VRAM comparisons will be limited.")


In [ ]:
def clear_memory():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()


def load_tokenizer(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_hf_model(model_name, dtype):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        trust_remote_code=True,
    )
    if DEVICE == "cuda":
        model = model.cuda()
    return model


def get_barqtrain_patch_fn(enable_cuda_backend):
    os.environ["BARQTRAIN_AUTO_BUILD"] = "1" if enable_cuda_backend else "0"
    os.environ["BARQTRAIN_VERBOSE_BUILD"] = "0"

    import barqtrain._ffi as ffi
    import barqtrain.ops as ops
    import barqtrain.lora as lora
    import barqtrain.patch_models as patch_models

    importlib.reload(ffi)
    importlib.reload(ops)
    importlib.reload(lora)
    patch_models = importlib.reload(patch_models)
    return patch_models.patch_model


def benchmark_generation(label, model, tokenizer, prompt, new_tokens, warmup_runs=2, measure_runs=5):
    model.eval()
    if hasattr(model, "config"):
        model.config.use_cache = True

    inputs = tokenizer(prompt, return_tensors="pt")
    if DEVICE == "cuda":
        inputs = {k: v.cuda() for k, v in inputs.items()}

    clear_memory()

    with torch.inference_mode():
        for _ in range(warmup_runs):
            _ = model.generate(**inputs, max_new_tokens=min(16, new_tokens), do_sample=False)

    latencies = []
    with torch.inference_mode():
        for _ in range(measure_runs):
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            outputs = model.generate(**inputs, max_new_tokens=new_tokens, do_sample=False)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            latencies.append(time.perf_counter() - t0)

    generated_tokens = outputs.shape[-1] - inputs["input_ids"].shape[-1]
    avg_latency = sum(latencies) / len(latencies)
    median_latency = statistics.median(latencies)
    peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2) if DEVICE == "cuda" else 0.0

    return {
        "setup": label,
        "dtype": str(DTYPE).replace("torch.", ""),
        "generated_tokens": generated_tokens,
        "avg_latency_s": avg_latency,
        "median_latency_s": median_latency,
        "tokens_per_sec": generated_tokens / avg_latency if avg_latency > 0 else 0.0,
        "peak_vram_mb": peak_vram_mb,
    }


def run_case(label, model_builder, tokenizer_builder):
    clear_memory()
    tokenizer = tokenizer_builder()
    model = model_builder()
    try:
        return benchmark_generation(label, model, tokenizer, PROMPT, NEW_TOKENS, WARMUP_RUNS, MEASURE_RUNS)
    finally:
        del model
        del tokenizer
        clear_memory()


In [ ]:
results = []

results.append(
    run_case(
        "baseline_hf_full_weight",
        model_builder=lambda: load_hf_model(BASE_MODEL_NAME, DTYPE),
        tokenizer_builder=lambda: load_tokenizer(BASE_MODEL_NAME),
    )
)

patch_model_no_cuda = get_barqtrain_patch_fn(enable_cuda_backend=False)
results.append(
    run_case(
        "barqtrain_patch_no_cuda_backend",
        model_builder=lambda: patch_model_no_cuda(load_hf_model(BASE_MODEL_NAME, DTYPE)),
        tokenizer_builder=lambda: load_tokenizer(BASE_MODEL_NAME),
    )
)

patch_model_with_cuda = get_barqtrain_patch_fn(enable_cuda_backend=True)
results.append(
    run_case(
        "barqtrain_patch_cuda_backend",
        model_builder=lambda: patch_model_with_cuda(load_hf_model(BASE_MODEL_NAME, DTYPE)),
        tokenizer_builder=lambda: load_tokenizer(BASE_MODEL_NAME),
    )
)

try:
    from unsloth import FastLanguageModel

    def build_unsloth_model():
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=UNSLOTH_MODEL_NAME,
            max_seq_length=1024,
            dtype=DTYPE,
            load_in_4bit=False,
        )
        FastLanguageModel.for_inference(model)
        if DEVICE == "cuda":
            model = model.cuda()
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        return model, tokenizer

    def unsloth_case():
        clear_memory()
        model, tokenizer = build_unsloth_model()
        try:
            return benchmark_generation(
                "unsloth_full_weight",
                model,
                tokenizer,
                PROMPT,
                NEW_TOKENS,
                WARMUP_RUNS,
                MEASURE_RUNS,
            )
        finally:
            del model
            del tokenizer
            clear_memory()

    results.append(unsloth_case())
except Exception as e:
    print("Unsloth benchmark skipped:", repr(e))


In [ ]:
df = pd.DataFrame(results)
df = df.sort_values("tokens_per_sec", ascending=False).reset_index(drop=True)
df


## Interpreting the benchmark

If BarqTrain is only modestly faster than baseline, that is expected with the current patch scope. The current codebase mainly patches RMSNorm. Recent performance work in the ecosystem shows that bigger gains usually come from attention, MLP, fused loss, and packing changes rather than RMSNorm alone.

High-value next steps to test after this notebook:
- FlashAttention-2 or FlashAttention-3 for the attention path.
- Sequence packing / padding-free batching to reduce wasted tokens.
- Fused linear + cross-entropy and more fused training kernels.
- More warmup runs when comparing against compiler-heavy stacks like Unsloth.


In [ ]:
out_path = "/content/barqtrain_benchmark_results.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)
